In [2]:
import pandas as pd

In [3]:
def process_dataset(file_path):
    # Load the dataset
    df = pd.read_csv(file_path)
    
    # Obtain unique PDBIDs
    unique_pdb_df = df.drop_duplicates(subset=['PDBID', 'Ligand Name'])
    
    # Separate the dataset into two based on the binding affinity measurement
    ic50_df = unique_pdb_df[unique_pdb_df['Binding Affinity Measurement'] == 'ic50'].reset_index(drop=True)
    ki_kd_df = unique_pdb_df[unique_pdb_df['Binding Affinity Measurement'].isin(['ki', 'kd'])].reset_index(drop=True)
    
    return ic50_df, ki_kd_df

In [4]:
# Example usage
ic50_data, ki_kd_data = process_dataset('hiqbind_sm_metadata.csv')

In [5]:
ic50_data = ic50_data[['Binding Affinity Value', 'Binding Affinity Unit', 'PDBID', 'Ligand Name', 'Ligand Chain', 'Ligand Residue Number']][:100]

In [6]:
ic50_data

,Binding Affinity Value,Binding Affinity Unit,PDBID,Ligand Name,Ligand Chain,Ligand Residue Number
0,0.004,uM,1a4g,ZMR,A,466
1,3.600,uM,1a4q,DPC,A,466
2,1100.000,nM,1ah0,SBI,A,320
3,16.000,nM,1ah3,TOL,A,320
4,47.000,uM,1am6,HAE,A,555
...,...,...,...,...,...,...
95,60.000,nM,1fvt,106,A,299
96,3.100,nM,1g05,BBH,B,901
97,7.000,nM,1g27,BB1,A,1001
98,10.000,nM,1g2a,BB2,A,1001


In [7]:
def verify_successful_optimization(row):
    pdbid = row['PDBID']
    ligand_name = row['Ligand Name']
    ligand_chain = row['Ligand Chain']
    ligand_residue_number = row['Ligand Residue Number']
    found_orca = False
    found_opt = False
    with open(f'{path_to_optimized_data}/{pdbid}/{pdbid}_{ligand_name}_{ligand_chain}_{ligand_residue_number}/GFN2-xTB/{pdbid}_{ligand_name}_{ligand_chain}_{ligand_residue_number}.out', 'r') as file:
        for line in file:   
            if "ORCA TERMINATED NORMALLY" in line:
                found_orca = True
            if "OPTIMIZATION RUN DONE" in line:
                found_opt = True

    if found_orca and found_opt:
        return True
    else:
        return False


In [8]:
path_to_optimized_data = os.environ.get('HIQBIND_DATA_DIR')  # Set HIQBIND_DATA_DIR to the raw HiQBind compound directory

# apply the verify_successful_optimization function to the ic50_data dataframe
ic50_data['successful_optimization'] = ic50_data.apply(verify_successful_optimization, axis=1)


In [9]:
ic50_data[['Binding Affinity Value', 'Binding Affinity Unit', 'PDBID', 'Ligand Name', 'Ligand Chain', 'Ligand Residue Number', 'successful_optimization']]

,Binding Affinity Value,Binding Affinity Unit,PDBID,Ligand Name,Ligand Chain,Ligand Residue Number,successful_optimization
0,0.004,uM,1a4g,ZMR,A,466,True
1,3.600,uM,1a4q,DPC,A,466,True
2,1100.000,nM,1ah0,SBI,A,320,True
3,16.000,nM,1ah3,TOL,A,320,True
4,47.000,uM,1am6,HAE,A,555,True
...,...,...,...,...,...,...,...
95,60.000,nM,1fvt,106,A,299,True
96,3.100,nM,1g05,BBH,B,901,True
97,7.000,nM,1g27,BB1,A,1001,True
98,10.000,nM,1g2a,BB2,A,1001,True


In [10]:
def filter_ic50_data(ic50_data):
    # Group by PDBID
    grouped = ic50_data.groupby('PDBID')
    print(grouped)
    
    # List to store filtered rows
    filtered_rows = []
    
    # Process each PDBID group
    for pdbid, group in grouped:
        # If only one entry for this PDBID, keep it
        if len(group) == 1:
            filtered_rows.append(group.iloc[0])
            continue
            
        # Filter for successful optimization
        successful_group = group[group['successful_optimization'] == True]
        
        # If none have successful optimization, skip
        if len(successful_group) == 0:
            continue
        
        # If only one has successful optimization, keep it
        if len(successful_group) == 1:
            filtered_rows.append(successful_group.iloc[0])
            continue
            
        # Check if all binding affinities are the same
        binding_values = successful_group['Binding Affinity Value'].unique()
        if len(binding_values) == 1:
            # Sort by ligand chain alphabetically and take the first one
            sorted_group = successful_group.sort_values('Ligand Chain')
            filtered_rows.append(sorted_group.iloc[0])
        else:
            # If binding affinities differ, we need to decide what to do
            # For now, let's choose the one with lowest chain alphabetically
            sorted_group = successful_group.sort_values('Ligand Chain')
            filtered_rows.append(sorted_group.iloc[0])
    
    # Create a new dataframe from the filtered rows
    filtered_ic50_data = pd.DataFrame(filtered_rows)
    return filtered_ic50_data

In [11]:
# Convert binding affinity values to log values based on units
def convert_to_log_value(row):
    value = row['Binding Affinity Value']
    unit = row['Binding Affinity Unit']
    
    # Convert to nanomolar if in micromolar
    if unit == 'uM':
        # 1 uM = 1000 nM
        value_in_nM = value * 1000
    elif unit == 'nM':
        value_in_nM = value
    else:
        # Handle other units or missing values
        return None
    
    # Calculate pIC50 (negative log of molar concentration)
    # Convert from nM to M (1 nM = 10^-9 M)
    molar_value = value_in_nM * 1e-9
    log_value = -1 * np.log10(molar_value)
    
    return log_value

In [48]:
filtered_ic50_data = filter_ic50_data(ic50_data)
filtered_ic50_data

,Binding Affinity Value,Binding Affinity Unit,PDBID,Ligand Name,Ligand Chain,Ligand Residue Number,successful_optimization,pIC50
0,0.004,uM,1a4g,ZMR,A,466,True,8.397940
1,3.600,uM,1a4q,DPC,A,466,True,5.443697
2,1100.000,nM,1ah0,SBI,A,320,True,5.958607
3,16.000,nM,1ah3,TOL,A,320,True,7.795880
4,47.000,uM,1am6,HAE,A,555,True,4.327902
...,...,...,...,...,...,...,...,...
95,60.000,nM,1fvt,106,A,299,True,7.221849
96,3.100,nM,1g05,BBH,B,901,True,8.508638
97,7.000,nM,1g27,BB1,A,1001,True,8.154902
98,10.000,nM,1g2a,BB2,A,1001,True,8.000000


In [ ]:
# Apply the conversion function
import numpy as np
ic50_data['pIC50'] = ic50_data.apply(convert_to_log_value, axis=1)

# Display the result
ic50_data[['Binding Affinity Value', 'Binding Affinity Unit', 'pIC50', 'Log Binding Affinity']]

In [24]:
# write the dataframes to csv files in pic50 and pki directories
ic50_data.to_csv('../pic50/organic_ic50.csv', index=False)
ki_kd_data.to_csv('../pki/organic_ki_kd.csv', index=False)


In [19]:
# load the dataframes from the csv files
ic50_data = pd.read_csv('../pic50/organic_pic50.csv')
ki_kd_data = pd.read_csv('../pk/organic_pk.csv')

In [23]:
ic50_data[ic50_data['successful_optimization'] == True]


,PDBID,Resolution,Year,Ligand Name,Ligand Chain,Ligand Residue Number,Binding Affinity Measurement,Binding Affinity Sign,Binding Affinity Value,Binding Affinity Unit,...,Ligand MW,Ligand LogP,Ligand TPSA,Ligand NumRotBond,Ligand NumHeavyAtoms,Ligand NumHDon,Ligand NumHAcc,Ligand QED,successful_optimization,pIC50
0,1a4g,2.2,1999,ZMR,A,466,ic50,=,0.004,uM,...,332.313,-6.7302,202.79,6,23,7,7,0.182040,True,8.397940
1,1a4q,1.9,1999,DPC,A,466,ic50,=,3.600,uM,...,389.452,-1.3845,126.41,8,28,2,5,0.566365,True,5.443697
2,1ah0,2.3,1998,SBI,A,320,ic50,=,1100.000,nM,...,236.202,0.6429,67.43,0,17,2,3,0.649534,True,5.958607
3,1ah3,2.3,1998,TOL,A,320,ic50,=,16.000,nM,...,356.345,2.2244,52.60,4,24,0,4,0.786655,True,7.795880
4,1am6,2.0,1998,HAE,A,555,ic50,=,47.000,uM,...,75.067,-0.4883,49.33,0,5,2,2,0.301333,True,4.327902
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6848,7vfw,NMR,2021,6I7,A,2420,ic50,=,16.000,nM,...,424.972,6.6525,36.28,4,30,0,4,0.469822,True,7.795880
6849,7vkn,2.7,2021,7GI,A,801,ic50,=,0.400,nM,...,355.373,2.5522,80.55,0,26,2,6,0.647765,True,9.397940
6850,7vko,2.9,2021,7GI,A,801,ic50,=,0.400,nM,...,355.373,2.5522,80.55,0,26,2,6,0.647765,True,9.397940
6851,7vsi,NMR,2021,7R3,A,703,ic50,=,3.100,nM,...,450.915,1.6134,108.61,6,31,4,7,0.529157,True,8.508638
